In [ ]:
# =============================================================================
# MULTI-SOURCE AI AGENT LAB - GEMINI RAG WORKER  (v4.0.0)
# Copy this entire cell into Google Colab.
# BACKEND_ID must match the `id` in src/services/colabPool.js
# =============================================================================

!pip install -q fastapi uvicorn pyngrok nest_asyncio pydantic rank_bm25 google-genai numpy

import os
import json
import asyncio
import time
import re
import nest_asyncio
import uvicorn
import numpy as np

from datetime import datetime, timezone
from typing import List, Dict, Any

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from pyngrok import ngrok
from google.colab import userdata
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types

# =============================================================================
# CONFIG
# =============================================================================

BACKEND_ID = "COLAB"          # MUST match colabPool.js `id`
BACKEND_VERSION = "4.0.0"
PORT = 8000

GEN_MODEL_NAME = "gemini-2.5-flash"
EMBED_MODEL_NAME = "gemini-embedding-001"

nest_asyncio.apply()

# Skip BM25 on pure code queries (CUST-*, ORD-*, TICK-*, POL-*, MAN-*)
CODE_RE = re.compile(r'^(CUST|ORD|TICK|POL|MAN|REC)-[A-Z0-9-]+$', re.I)

# =============================================================================
# SECRETS
# =============================================================================

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not NGROK_TOKEN:
    raise RuntimeError("Missing NGROK_TOKEN in Colab Secrets.")
if not GEMINI_API_KEY:
    raise RuntimeError("Missing GEMINI_API_KEY in Colab Secrets.")

ngrok.set_auth_token(NGROK_TOKEN)

client = genai.Client(api_key=GEMINI_API_KEY)
SERVER_START_TIME = time.time()

# =============================================================================
# LOCAL FALLBACK KNOWLEDGE
# This is only used for soft keyword hints. The authoritative evidence
# comes from the Node side inside payload.Evidence.
# =============================================================================

KNOWLEDGE_BASE: List[Dict[str, Any]] = [
    {
        "id": "pol_001",
        "category": "returns",
        "title": "Returns & Refunds Policy",
        "content": "Unused standard items in original packaging can be returned within 30 days with proof of purchase. Custom-cut timber and tinted paint are non-refundable for change of mind. Faulty items are subject to applicable Australian Consumer Law requirements."
    },
    {
        "id": "prd_101",
        "category": "timber",
        "title": "H2 Blue Structural Pine Specifications",
        "content": "H2 Blue structural pine is intended for protected interior framing applications. It is not suitable for outdoor exposed or ground-contact use."
    },
    {
        "id": "prd_102",
        "category": "timber",
        "title": "H3 & H4 Outdoor Treated Pine",
        "content": "H3 treated pine is designed for above-ground outdoor applications. H4 treated pine is suitable for direct ground-contact applications."
    },
    {
        "id": "prd_103",
        "category": "tools",
        "title": "Power Tool Warranty & Servicing",
        "content": "Defective power tools returned within 30 days may qualify for direct exchange or refund. Beyond 30 days, authorised service inspection may be required."
    },
    {
        "id": "manual_001",
        "category": "tools",
        "title": "Makita Drill Manual Summary",
        "content": "The Makita 18V brushless cordless drill is intended for drilling and screwdriving applications. Follow manufacturer operating and safety instructions."
    }
]

bm25_index = BM25Okapi([
    doc["content"].lower().split()
    for doc in KNOWLEDGE_BASE
])

kb_embeddings: List[np.ndarray] = []
EMBEDDINGS_AVAILABLE = True

def embed_text(text: str, task_type: str) -> np.ndarray:
    result = client.models.embed_content(
        model=EMBED_MODEL_NAME,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return np.asarray(result.embeddings[0].values, dtype=np.float32)

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def normalize(scores):
    scores = np.asarray(scores, dtype=np.float32)
    if scores.size == 0:
        return scores
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-9:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)

try:
    for doc in KNOWLEDGE_BASE:
        kb_embeddings.append(embed_text(doc["content"], "RETRIEVAL_DOCUMENT"))
    print("Dense policy/document embeddings initialized.")
except Exception as exc:
    EMBEDDINGS_AVAILABLE = False
    kb_embeddings = []
    print(f"Embedding index unavailable; BM25-only fallback: {exc}")


def retrieve_local_kb(query: str, top_k: int = 4):
    # Code queries should never hit BM25 — they are resolved by Node via
    # the Evidence packet. Returning [] prevents unrelated policy docs
    # from being injected into the prompt.
    if CODE_RE.match(query.strip()):
        return []

    tokens = query.lower().split()
    bm25_scores = normalize(bm25_index.get_scores(tokens))
    combined = bm25_scores.copy()

    if EMBEDDINGS_AVAILABLE:
        try:
            q = embed_text(query, "RETRIEVAL_QUERY")
            semantic = normalize([
                cosine(q, emb)
                for emb in kb_embeddings
            ])
            combined = 0.5 * bm25_scores + 0.5 * semantic
        except Exception as exc:
            print(f"Semantic query failed; BM25-only: {exc}")

    indices = np.argsort(combined)[::-1][:top_k]
    return [
        KNOWLEDGE_BASE[i]
        for i in indices
        if combined[i] > 0
    ]

# =============================================================================
# API MODELS
# =============================================================================

class ChatPayload(BaseModel):
    Query: str = Field(..., min_length=1, max_length=5000)
    Evidence: Dict[str, Any] = Field(default_factory=dict)
    ContractVersion: str = Field(default="1.0")
    RequestId: str = Field(default="")

# =============================================================================
# FASTAPI
# =============================================================================

app = FastAPI(
    title="Multi-Source Hybrid RAG Worker",
    version=BACKEND_VERSION
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/health")
async def health():
    ready = bool(GEN_MODEL_NAME and KNOWLEDGE_BASE)
    return {
        "status": "healthy" if ready else "degraded",
        "backend_id": BACKEND_ID,
        "version": BACKEND_VERSION,
        "contract_version": "1.0",
        "ready_for_chat": ready,
        "generation_model": GEN_MODEL_NAME,
        "embedding_model": EMBED_MODEL_NAME,
        "policy_documents": len(KNOWLEDGE_BASE),
        "policy_embeddings": EMBEDDINGS_AVAILABLE,
        "uptime_seconds": int(time.time() - SERVER_START_TIME),
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    }

@app.post("/api/chat")
async def chat(payload: ChatPayload):
    query = payload.Query.strip()
    if not query:
        raise HTTPException(status_code=400, detail="Query cannot be empty.")

    print(
        f"[{payload.RequestId}] /api/chat "
        f"query={query!r} "
        f"evidence_customers={len(payload.Evidence.get('customers', []))} "
        f"evidence_orders={len(payload.Evidence.get('orders', []))} "
        f"evidence_policies={len(payload.Evidence.get('policies', []))} "
        f"evidence_tickets={len(payload.Evidence.get('tickets', []))}"
    )

    try:
        local_docs = retrieve_local_kb(query)
        external_evidence = payload.Evidence or {}

        local_context = "\n\n".join(
            f"ID: {d['id']}\nTYPE: {d['category']}\nTITLE: {d['title']}\nCONTENT: {d['content']}"
            for d in local_docs
        ) or "No worker-local knowledge matched."

        evidence_json = json.dumps(
            external_evidence,
            ensure_ascii=False,
            default=str,
            indent=2
        )

        prompt = f"""
You are a domain-specific AI systems specialist operating inside a multi-source evidence architecture.

USER QUERY:
{query}

WORKER-LOCAL KNOWLEDGE (supporting, not authoritative):
{local_context}

NODE / EXTERNAL EVIDENCE (authoritative for customer/order/ticket/policy facts):
{evidence_json}

Rules:
1. Treat supplied NODE / EXTERNAL EVIDENCE as authoritative for the scope it represents.
2. PostgreSQL transaction records are facts about the current lab dataset.
3. Static KB / local knowledge is supporting reference only.
4. Do not invent a customer, order, product, ticket, policy or API result.
5. Do not treat missing evidence as evidence of absence unless the source explicitly says so.
6. Separate transactional facts from general policy/manual information.
7. If evidence is insufficient, set action_status to INFORMATION NOT FOUND.
8. If the user query is a bare code like CUST-001, ORD-1001, TICK-001, POL-RET-001 or MAN-MAK-001,
   match it against the corresponding *_code field of the records in NODE / EXTERNAL EVIDENCE
   before deciding nothing was found.
9. A record whose *_code field equals the query IS the matching record.
10. Never say "no record found" if a record with a matching *_code field is present in evidence.
11. Give concise verification steps, not hidden chain-of-thought.
12. Return ONLY valid JSON. No markdown.

Required JSON:
{{
  "action_status": "string",
  "reasoning_steps": ["string"],
  "final_answer": "string",
  "customer_next_steps": ["string"],
  "evidence": {{
    "knowledge_ids": ["string"],
    "order_ids": ["string"],
    "ticket_ids": ["string"]
  }}
}}
"""

        response = client.models.generate_content(
            model=GEN_MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
                response_mime_type="application/json"
            )
        )

        raw_text = response.text or ""

        try:
            parsed = json.loads(raw_text)
        except Exception:
            parsed = {
                "action_status": "ASSISTANCE REQUIRED",
                "reasoning_steps": [],
                "final_answer": raw_text,
                "customer_next_steps": [],
                "evidence": {
                    "knowledge_ids": [d["id"] for d in local_docs],
                    "order_ids": [],
                    "ticket_ids": []
                }
            }

        return {
            "status": "success",
            "backend_id": BACKEND_ID,
            "data": {
                "query": query,
                "contexts_used": [d["id"] for d in local_docs],
                "response": json.dumps(parsed),
                "local_retrieval_count": len(local_docs),
                "external_evidence_received": bool(external_evidence)
            }
        }

    except Exception as exc:
        print("CHAT ERROR:", repr(exc))
        raise HTTPException(
            status_code=500,
            detail=f"RAG processing failed: {exc}"
        )

# =============================================================================
# START
# =============================================================================

def run_server():
    ngrok.kill()
    tunnel = ngrok.connect(PORT, proto="http")
    url = tunnel.public_url

    print("\n" + "=" * 80)
    print(f"BACKEND ID : {BACKEND_ID}")
    print(f"BACKEND URL: {url}")
    print("=" * 80)
    print(f"HEALTH     : {url}/health")
    print(f"CHAT       : {url}/api/chat")
    print(f"DOCS       : {url}/docs")
    print("=" * 80 + "\n")

    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=PORT,
        loop="asyncio",
        log_level="info"
    )
    server = uvicorn.Server(config)
    asyncio.get_event_loop().run_until_complete(server.serve())

run_server()